In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

path = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\AccuracyMeans.csv"
df = pd.read_csv(path)

id_vars = ['participant','gender','negative_affect','positive_affect']
value_vars = [c for c in df.columns if c not in id_vars]
accuracyLong = pd.melt(df, id_vars=id_vars, value_vars=value_vars, var_name='emotion', value_name='value')
accuracyLong['value'] = pd.to_numeric(accuracyLong['value'], errors='coerce')
accuracyLong['emotion'] = accuracyLong['emotion'].astype(str).str.replace(r'_acc$','',regex=True).str.capitalize()
stats = accuracyLong.groupby('emotion', observed=True)['value'].agg(['mean','std','count']).reset_index()

# Turn this into a dataframe
stats_df = stats[['emotion','mean','std']]

# Rename columns
stats_df = stats_df.rename(columns={'emotion': 'Expression', 'mean': 'Mean', 'std': 'S.D.'})

In [3]:
# Path to the RADIATE CSV file in this workspace
path_radiate = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\RADIATE_mean_prop_correct.csv"
radiate = pd.read_csv(path_radiate)

# Remove all the open expressions except for angry and fear and surprise
radiate = radiate[~radiate['Expression'].str.contains('open') | radiate['Expression'].str.contains('Angry|Fear|Surprise')]

# remove rows 0 and 6
radiate = radiate.drop([0, 2, 6, 9])

# now remove the open\closed strings from the Expression column
radiate['Expression'] = radiate['Expression'].str.replace(' (open)', '').str.replace(' (closed)', '').str.strip()

In [13]:
# Combine the two dataframes (stats_df from AccuracyMeans and radiate from RADIATE file)
# Use descriptive suffixes instead of the default _x/_y so it's clear which dataset each column came from
combined = pd.merge(stats_df, radiate, on='Expression', how='inner', suffixes=('_UBIVFED','_RADIATE'))

# Outer merge with indicator to list any expressions that exist in only one dataframe (also use same suffixes)
merged_with_indicator = pd.merge(stats_df, radiate, on='Expression', how='outer', indicator=True, suffixes=('_UBIVFED','_RADIATE'))
left_only = merged_with_indicator[merged_with_indicator['_merge'] == 'left_only'][['Expression']]
right_only = merged_with_indicator[merged_with_indicator['_merge'] == 'right_only'][['Expression']]

# Replace the Happy expression to Joy (post-merge to affect both datasets)
combined.loc[combined['Expression'] == 'Happy', 'Expression'] = 'Joy'

# rename Sad to Sadness
combined.loc[combined['Expression'] == 'Sad', 'Expression'] = 'Sadness'

# rename Angry to Anger
combined.loc[combined['Expression'] == 'Angry', 'Expression'] = 'Anger'

# reorder the expressions in a specific order: Neutral, Joy, Sadness, Anger, Fear, Surprise, Disgust
desired_order = ['Neutral', 'Joy', 'Sadness', 'Anger', 'Fear', 'Surprise', 'Disgust']
combined['Expression'] = pd.Categorical(combined['Expression'], categories=desired_order, ordered=True)
combined = combined.sort_values('Expression').reset_index(drop=True)

# Identify expected mean/sd columns after merge
mean_col_ub = 'Mean_UBIVFED'
sd_col_ub = 'S.D._UBIVFED'
mean_col_rad = 'Mean_RADIATE'
sd_col_rad = 'S.D._RADIATE'

# Round numerical columns (means & sds) for consistent formatting
for c in [mean_col_ub, sd_col_ub, mean_col_rad, sd_col_rad]:
    if c in combined.columns:
        combined[c] = combined[c].astype(float).round(2)

# Build display columns with "mean (sd)" formatting
combined['UBIVFED'] = combined.apply(lambda r: f"{r[mean_col_ub]:.2f} ({r[sd_col_ub]:.2f})", axis=1)
combined['RADIATE'] = combined.apply(lambda r: f"{r[mean_col_rad]:.2f} ({r[sd_col_rad]:.2f})", axis=1)

# Row-wise mean accuracy (average of means only)
combined['Mean accuracy'] = ((combined[mean_col_ub] + combined[mean_col_rad]) / 2).round(2)
combined['Mean accuracy'] = combined['Mean accuracy'].map(lambda v: f"{v:.2f}")

# Create final table with only required columns
final_table = combined[['Expression','UBIVFED','RADIATE','Mean accuracy']].copy()
# Rename UBIVFED/RADIATE columns to requested headers
final_table = final_table.rename(columns={'UBIVFED':'Mean (sd) UBIVFED','RADIATE':'Mean (sd) RADIATE'})

# Bottom summary row: overall mean (sd) per dataset + overall mean accuracy
overall_mean_ub = combined[mean_col_ub].mean()
overall_sd_ub = 0.076
overall_mean_rad = combined[mean_col_rad].mean()
overall_sd_rad = 0.19
overall_mean_accuracy = ((overall_mean_ub + overall_mean_rad) / 2)

summary_row = {
    'Expression': 'Overall',
    'Mean (sd) UBIVFED': f"{overall_mean_ub:.2f} ({overall_sd_ub:.2f})",
    'Mean (sd) RADIATE': f"{overall_mean_rad:.2f} ({overall_sd_rad:.2f})",
    'Mean accuracy': f"{overall_mean_accuracy:.2f}"
}
final_table = pd.concat([final_table, pd.DataFrame([summary_row])], ignore_index=True)

# Convert to LaTeX (no index)
latex_table = final_table.to_latex(index=False,
                                   caption='Comparison of Emotion Recognition Accuracy: UBIVFED vs RADIATE',
                                   label='tab:accuracy_comparison',
                                   position='h')

# Remove underscores for cleaner appearance
latex_table = latex_table.replace('_', ' ')

# Save LaTeX table to file
output_path = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\accuracy_comparison_table.tex"
with open(output_path, 'w') as f:
    f.write(latex_table)

In [ ]:
# read the comparison CSV (path from the original R script)
path_comp = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\Master Compiled Doc - Cleaned - Comparisons.csv"
comparison = pd.read_csv(path_comp)

# split by condition
comparison_CC = comparison[comparison['Condition'] == 'CC']
comparison_IC = comparison[comparison['Condition'] == 'IC']

# run a pairwise t-test between the two conditions for each emotion
emotions = comparison['RealEmotionID'].unique()
rows = []
for emotion in emotions:
    data_CC = comparison_CC[comparison_CC['RealEmotionID'] == emotion]['slider.response'].dropna()
    data_IC = comparison_IC[comparison_IC['RealEmotionID'] == emotion]['slider.response'].dropna()

    # Student's t-test (equal variances) to match CI calc below
    t_stat, p_value = stats.ttest_ind(data_CC, data_IC, nan_policy='omit', equal_var=True)

    mean_CC = data_CC.mean()
    mean_IC = data_IC.mean()
    mean_diff = mean_CC - mean_IC

    n1, n2 = data_CC.size, data_IC.size
    
    # compute 95% CI for CC and IC separately
    if n1 > 1:
        s1 = data_CC.std(ddof=1)
        df1 = n1 - 1
        se_CC = s1 / np.sqrt(n1)
        t_crit_CC = stats.t.ppf(0.975, df1)
        cc_ci_err = t_crit_CC * se_CC
    else:
        cc_ci_err = np.nan
    
    if n2 > 1:
        s2 = data_IC.std(ddof=1)
        df2 = n2 - 1
        se_IC = s2 / np.sqrt(n2)
        t_crit_IC = stats.t.ppf(0.975, df2)
        ic_ci_err = t_crit_IC * se_IC
    else:
        ic_ci_err = np.nan
    
    # compute pooled-variance standard error and 95% CI for mean difference
    if n1 > 1 and n2 > 1:
        df = n1 + n2 - 2
        sp2 = ((n1 - 1) * (s1 ** 2) + (n2 - 1) * (s2 ** 2)) / df
        se_diff = np.sqrt(sp2 * (1.0 / n1 + 1.0 / n2))
        t_crit = stats.t.ppf(0.975, df)
        ci_lower = mean_diff - t_crit * se_diff
        ci_upper = mean_diff + t_crit * se_diff
    else:
        df = np.nan
        ci_lower = np.nan
        ci_upper = np.nan

    rows.append({
        'Expression': emotion,
        't_stat': t_stat,
        'p_value': p_value,
        'mean_CC': mean_CC,
        'mean_IC': mean_IC,
        'mean_diff': mean_diff,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'cc_ci_err': cc_ci_err,
        'ic_ci_err': ic_ci_err
    })

# Build final DataFrame with a clearer name
ttest_emotion_summary = pd.DataFrame(rows)

# reorder the expressions: Neutral, Joy, Sadness, Anger, Fear, Surprise, Disgust
expression_order = ['Neutral', 'Happy', 'Sad', 'Angry', 'Fear', 'Surprise', 'Disgust']
ttest_emotion_summary['Expression'] = pd.Categorical(ttest_emotion_summary['Expression'], categories=expression_order, ordered=True)
ttest_emotion_summary = ttest_emotion_summary.sort_values('Expression')

# rename Happy to Joy, Angry to Anger, Sad to Sadness
ttest_emotion_summary['Expression'] = ttest_emotion_summary['Expression'].replace({'Happy': 'Joy', 'Angry': 'Anger', 'Sad': 'Sadness'})

df_plot = ttest_emotion_summary.copy()
df_plot['sig_label'] = df_plot['p_value'].apply(lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
x = np.arange(len(df_plot))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - bar_width / 2, df_plot['mean_CC'], bar_width, label='C', color='#4DBBD5', 
               yerr=df_plot['cc_ci_err'], capsize=4, error_kw={'elinewidth': 1.5})
bars2 = ax.bar(x + bar_width / 2, df_plot['mean_IC'], bar_width, label='I', color='#DC0000',
               yerr=df_plot['ic_ci_err'], capsize=4, error_kw={'elinewidth': 1.5})

tops = np.maximum(
    df_plot['mean_CC'] + df_plot['cc_ci_err'].fillna(0),
    df_plot['mean_IC'] + df_plot['ic_ci_err'].fillna(0)
 )
text_y = tops + 0.12
for xi, label, y in zip(x, df_plot['sig_label'], text_y):
    if label:
        ax.text(xi, y, label, ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(df_plot['Expression'], fontsize=14)
ax.tick_params(axis='y', labelsize=14)
ax.set_ylabel('Mean slider response', fontsize=16)
y_max = max(text_y.max() + 0.2, 7)
ax.set_ylim(4, y_max)
ax.set_xlabel('Emotion', fontsize=16)
ax.set_title('Congruent (C) vs Incongruent (I) Conditions by Emotion', fontsize=20)
ax.legend(loc='upper right', frameon=True, fontsize=14)
fig.tight_layout()
plt.savefig(r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\plots\zedd_figures\similarity_means_grouped_plot.png", dpi=300)
plt.close()

C:\Users\super\AppData\Local\Temp\ipykernel_6612\1058498988.py:83: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  ttest_emotion_summary['Expression'] = ttest_emotion_summary['Expression'].replace({'Happy': 'Joy', 'Angry': 'Anger', 'Sad': 'Sadness'})
